### 1. Environment Setup & Discovery Crawler
Initializes folder structures (`data/raw/html_resorts/` and `data/processed/`) and executes a multi-page discovery crawler across national indices (CH, IT, AT, FR) to gather ~75 major ski resorts per country (~300 total) adhering to polite request intervals (2.5s) and academic User-Agent identification.

In [7]:
import os
import re
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 1. Directory Setup
RAW_HTML_DIR = "data/raw/html_resorts"
PROCESSED_DIR = "data/processed"
os.makedirs(RAW_HTML_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# 2. Ethical Scraping Settings
HEADERS = {
    "User-Agent": "ZHAW-DataAnalytics-Research-StudentProject/1.0 (contact: student@zhaw.ch)",
    "Accept-Language": "en-US,en;q=0.9"
}
REQUEST_DELAY_SECONDS = 2.0
BASE_URL = "https://www.skiresort.info"

COUNTRIES = [
    {"code": "CH", "path": "/ski-resorts/switzerland/"},
    {"code": "IT", "path": "/ski-resorts/italy/"},
    {"code": "AT", "path": "/ski-resorts/austria/"},
    {"code": "FR", "path": "/ski-resorts/france/"}
]

resorts_to_scrape = []
print("--> Starting automated URL discovery across Switzerland, Italy, Austria, and France...\n")

for c in COUNTRIES:
    c_code = c["code"]
    country_collected = 0
    seen_urls = set()
    print(f"Collecting resorts for {c_code}...")

    # Traversa le prime 4 pagine dell'indice nazionale
    for page_num in range(1, 5):
        if country_collected >= 75:
            break

        if page_num == 1:
            page_url = f"{BASE_URL}{c['path']}"
        else:
            page_url = f"{BASE_URL}{c['path']}page/{page_num}/"

        try:
            res = requests.get(page_url, headers=HEADERS, timeout=15)
            if res.status_code == 200:
                soup = BeautifulSoup(res.text, "html.parser")
                
                # Cerca tutti i link che puntano a una scheda di comprensorio (/ski-resort/...)
                for a in soup.find_all("a", href=True):
                    href = a["href"].strip()
                    
                    # Normalizza a URL assoluto se relativo
                    if href.startswith("/"):
                        full_resort_url = f"{BASE_URL}{href}"
                    else:
                        full_resort_url = href

                    # Verifica che sia un link valido a un comprensorio
                    if "/ski-resort/" in full_resort_url and not any(sub in full_resort_url for sub in [
                        "page/", "search", "sorted", "resorts", "regions", 
                        "test-result", "trail-map", "photos", "weather", "webcams", "hotel"
                    ]):
                        # Rimuove parametri query o slash finali per deduplica
                        clean_url = full_resort_url.split("?")[0].rstrip("/") + "/"
                        
                        # Verifica che il link abbia la struttura canonica: .../ski-resort/{nome}/
                        parts = [p for p in clean_url.split("/") if p]
                        if len(parts) >= 4 and parts[-2] == "ski-resort":
                            if clean_url not in seen_urls:
                                name = a.get_text().strip()
                                # Se il link non ha testo diretto, ricava il nome dallo slug URL
                                if len(name) < 3 or "\n" in name:
                                    name = parts[-1].replace("-", " ").title()

                                seen_urls.add(clean_url)
                                resorts_to_scrape.append({
                                    "name": name,
                                    "country": c_code,
                                    "url": clean_url
                                })
                                country_collected += 1
                                if country_collected >= 75:
                                    break
            time.sleep(REQUEST_DELAY_SECONDS)
        except Exception as e:
            print(f"  [WARN] Issue fetching {page_url}: {e}")
            break

    print(f"  [OK] Found {country_collected} valid resorts for {c_code}.")

print("\n" + "=" * 65)
print(f"CRAWLER SUMMARY: Collected {len(resorts_to_scrape)} target resort URLs.")
print("=" * 65)

--> Starting automated URL discovery across Switzerland, Italy, Austria, and France...

  [OK] Found 75 valid resorts for CH.
  [OK] Found 75 valid resorts for IT.
  [OK] Found 75 valid resorts for AT.
  [OK] Found 75 valid resorts for FR.

CRAWLER SUMMARY: Collected 300 target resort URLs.


### 2. Main HTML Caching & Variable Extraction Loop
Iterates through discovered destination URLs, caching raw HTML to `data/raw/html_resorts/` to prevent redundant network calls. Uses BeautifulSoup and NFKD unicode normalization to extract elevations (base and top in meters), slope kilometers, lifts count, adult high-season prices, star ratings, and artificial snow cannons.

In [12]:
scraped_records = []
print(f"--> Re-parsing {len(resorts_to_scrape)} local HTML files with calibrated patterns...\n")

for idx, item in enumerate(resorts_to_scrape):
    # Sanitize URL slug to locate cached files on disk
    url_slug = [p for p in item["url"].split("/") if p][-1]
    slug = re.sub(r"[^a-z0-9]+", "_", url_slug.lower())[:60]
    raw_html_path = os.path.join(RAW_HTML_DIR, f"{slug}.html")

    # Fallback to full name slug if cached under the previous naming scheme
    if not os.path.exists(raw_html_path):
        old_slug = re.sub(r"[^a-z0-9]+", "_", item["name"].lower().strip("_"))
        raw_html_path = os.path.join(RAW_HTML_DIR, f"{old_slug}.html")

    if not os.path.exists(raw_html_path):
        continue

    # Read from local disk cache (zero network calls)
    with open(raw_html_path, "r", encoding="utf-8") as f:
        html_doc = f.read()

    soup = BeautifulSoup(html_doc, "html.parser")
    full_text = unicodedata.normalize("NFKD", soup.get_text(separator=" "))

    # 1. Base and Top Elevation (meters)
    elevation_base_m, elevation_top_m = None, None
    elev_match = re.search(r"(\d{3,4})\s*m\s*[-–—/]\s*(\d{3,4})\s*m", full_text)
    if not elev_match:
        elev_match = re.search(r"(\d{1,2}[,\.\s]\d{3}|\d{3,4})\s*m\s*[-–—/]\s*(\d{1,2}[,\.\s]\d{3}|\d{3,4})\s*m", full_text)
    if elev_match:
        b = int(re.sub(r"[^\d]", "", elev_match.group(1)))
        t = int(re.sub(r"[^\d]", "", elev_match.group(2)))
        if 300 <= b <= 4000 and 400 <= t <= 4500 and b < t:
            elevation_base_m, elevation_top_m = b, t

    # 2. Total Slope Length (km)
    slope_km = None
    slope_match = re.search(r"(?:Ski slopes|Slopes?|Pistes?|Total)[\s\w:]*?(\d+(?:[\.,]\d+)?)\s*km", full_text, re.IGNORECASE)
    if not slope_match:
        slope_match = re.search(r"(\d+(?:[\.,]\d+)?)\s*km\b", full_text)
    if slope_match:
        try:
            val = float(slope_match.group(1).replace(",", "."))
            if 1.0 <= val <= 650.0:
                slope_km = val
        except ValueError:
            pass

    # 3. Lifts Count
    lifts_count = None
    lifts_match = re.search(r"(\d+)\s*(?:ski lifts|lifts|lift)", full_text, re.IGNORECASE)
    if lifts_match:
        l_cnt = int(lifts_match.group(1))
        if 1 <= l_cnt <= 200:
            lifts_count = l_cnt

    # 4. Adult Ticket Price & Currency
    price_day_raw = None
    currency = "CHF" if item["country"] == "CH" else "EUR"
    price_match = re.search(r"(?:Adults?|Day ticket|Day pass|Skipass|1-day)[^\d\n\r]{0,40}(?:SFr\.?|€|CHF|EUR)?\s*(\d{2,3}(?:[,\.]\d{1,2})?)\s*(?:,-|\.-)?", full_text, re.IGNORECASE)
    if not price_match:
        price_match = re.search(r"(?:SFr\.?|CHF|€)\s*(\d{2,3}(?:[,\.]\d{1,2})?)\s*(?:,-|\.-)?", full_text)
    if price_match:
        raw_p = price_match.group(1).replace(",", ".").rstrip(".")
        try:
            val = float(re.sub(r"[^\d\.]", "", raw_p))
            if 15.0 <= val <= 180.0:
                price_day_raw = val
        except ValueError:
            pass

    # 5. Rating Stars (out of 5.0)
    rating_stars = None
    star_tag = soup.find(attrs={"title": re.compile(r"out of 5 stars|of 5 stars", re.I)})
    if star_tag:
        m = re.search(r"(\d(?:\.\d)?)", star_tag.get("title", ""))
        if m:
            rating_stars = float(m.group(1))

    # 6. Artificial Snowmaking (Cannons)
    snow_cannons = None
    cannons_m = re.search(r"(\d+)\s*(?:snow cannons|snowmakers|snow making|cannons)", full_text, re.IGNORECASE)
    if cannons_m:
        snow_cannons = int(cannons_m.group(1))

    scraped_records.append({
        "resort_name": item["name"],
        "country": item["country"],
        "url": item["url"],
        "elevation_base_m": elevation_base_m,
        "elevation_top_m": elevation_top_m,
        "slope_km": slope_km,
        "lifts_count": lifts_count,
        "price_day_raw": price_day_raw,
        "currency": currency,
        "rating_stars": rating_stars,
        "snow_cannons": snow_cannons
    })

print(f"--> Re-parsing completed successfully for {len(scraped_records)} ski resorts!")


--> Re-parsing 300 local HTML files with calibrated patterns...

--> Re-parsing completed successfully for 300 ski resorts!


### 3. Price Harmonization (EUR), Cleansing & Export
Harmonizes ticket prices into a single continuous variable (`price_day_eur`) using the benchmark ECB conversion factor ($1\text{ CHF} \approx 1.05\text{ EUR}$), imputes missing artificial snowmaker counts with 0, removes observations missing essential morphological variables, and saves the final dataset to `data/processed/resorts_clean.csv`.

In [13]:
df_raw = pd.DataFrame(scraped_records)

# 1. Currency Harmonization into EUR (1 CHF ≈ 1.05 EUR benchmark)
CHF_TO_EUR_RATE = 1.05

def convert_to_eur(row):
    if pd.isna(row["price_day_raw"]):
        return None
    if row["currency"] == "CHF":
        return round(row["price_day_raw"] * CHF_TO_EUR_RATE, 2)
    return round(row["price_day_raw"], 2)

df_raw["price_day_eur"] = df_raw.apply(convert_to_eur, axis=1)

# 2. Retain resorts with valid core morphological data (base and top elevation)
df_clean = df_raw.dropna(subset=["elevation_base_m", "elevation_top_m"]).copy()

# 3. Missing Value Imputation (Data Cleansing Best Practices)
# Impute snow cannons with 0 (no mechanical installation reported)
df_clean["snow_cannons"] = df_clean["snow_cannons"].fillna(0).astype(int)

# Impute missing slope kilometers and day pass prices using national medians
df_clean["slope_km"] = df_clean["slope_km"].fillna(
    df_clean.groupby("country")["slope_km"].transform("median")
).round(1)

df_clean["price_day_eur"] = df_clean["price_day_eur"].fillna(
    df_clean.groupby("country")["price_day_eur"].transform("median")
).round(2)

# 4. Export Cleaned Dataset to CSV
OUTPUT_CLEAN_CSV = os.path.join(PROCESSED_DIR, "resorts_clean.csv")
df_clean.to_csv(OUTPUT_CLEAN_CSV, index=False)

print("=" * 80)
print(f"CLEANED DATASET EXPORTED TO: {OUTPUT_CLEAN_CSV}")
print("=" * 80)
print(f"Total valid alpine resorts retained: {len(df_clean)}")
print("\nBreakdown by Country:")
print(df_clean["country"].value_counts())
print("\nDescriptive Summary Statistics:")
print(df_clean[["elevation_base_m", "elevation_top_m", "slope_km", "price_day_eur", "rating_stars"]].describe())

CLEANED DATASET EXPORTED TO: data/processed/resorts_clean.csv
Total valid alpine resorts retained: 300

Breakdown by Country:
country
CH    75
IT    75
AT    75
FR    75
Name: count, dtype: int64

Descriptive Summary Statistics:
       elevation_base_m  elevation_top_m    slope_km  price_day_eur  rating_stars
count        300.000000       300.000000  300.000000     300.000000    300.000000
mean        1249.740000      2412.103333   57.123333      64.402833      3.803333
std          341.182361       454.901054   80.645680      16.364740      0.504815
min          509.000000      1338.000000    1.000000      24.000000      2.700000
25%         1002.250000      2093.750000    9.750000      52.000000      3.300000
50%         1232.500000      2375.000000   32.350000      66.075000      3.800000
75%         1477.500000      2714.750000   63.250000      75.000000      4.200000
max         2760.000000      3899.000000  580.000000     121.800000      4.900000
